# 11.5 - Vector Databases

**Phase:** 11 - RAG Systems

**Status:** VERIFIED

---

## 1. What Are We Solving?

Vector databases store, index, and search high-dimensional vectors efficiently using approximate nearest neighbor (ANN) algorithms. Linear scan is too slow at millions of vectors; HNSW and IVF trade a little accuracy for huge speedups.

## 2. Why Does This Matter?

This is the infrastructure layer of RAG. Chroma is great for local prototyping, FAISS for high-performance in-memory search, Pinecone/Qdrant for production. You need to store vectors, query them, and filter by metadata.

## 3. Prerequisites

Units 11.3 (Embeddings), 11.4 (Vector Similarity).

## 4. Learning Objectives

By the end of this unit, you should be able to:
- Store + query arbitrary vectors in Chroma (ephemeral) with metadata filters
- Store + query vectors in FAISS IndexFlatIP (in-memory)
- Compare the two and explain ANN (HNSW / IVF) trade-offs

## 5. Mental Model

A vector database is a search engine for meaning: instead of indexing words it indexes vectors and finds the closest ones to a query.

```text
Naive: O(n*d) per query  - scan every vector
HNSW:  O(log n)          - navigate a graph
IVF:   O(n_lists*d)      - search only relevant partitions
```


## 6. Chroma: Store + Query Vectors
`chromadb.Client()` is ephemeral (in-memory). We pass `embedding_function=None` and supply our own vectors so no model is downloaded. We store vectors + documents + metadata and query by a vector.

In [1]:
import chromadb, numpy as np

client = chromadb.Client()
col = client.create_collection("vecs_demo", embedding_function=None,
                               metadata={"hnsw:space": "cosine"})
col.add(
    documents=["return policy is 30 days", "shipping takes 5-7 days", "refunds in 7 days",
               "express shipping 2 days"],
    embeddings=[[1.0, 0.0, 0.1], [0.2, 0.9, 0.1], [0.95, 0.1, 0.0], [0.1, 0.9, 0.0]],
    ids=["d1", "d2", "d3", "d4"],
    metadatas=[{"topic": "returns"}, {"topic": "shipping"}, {"topic": "returns"}, {"topic": "shipping"}],
)
print("count:", col.count())

res = col.query(query_embeddings=[[0.9, 0.1, 0.0]], n_results=2)
for doc, dist, meta in zip(res["documents"][0], res["distances"][0], res["metadatas"][0]):
    print(f"dist={dist:.3f} topic={meta['topic']} | {doc}")


count: 4
dist=0.000 topic=returns | refunds in 7 days
dist=0.011 topic=returns | return policy is 30 days


## 7. Chroma: Metadata Filter
Narrow the search space with a `where` filter before ranking, e.g. only 'shipping' docs. This rejects unrelated documents outright.

In [2]:
res = col.query(query_embeddings=[[0.9, 0.1, 0.0]], n_results=3, where={"topic": "shipping"})
print("filtered to shipping:")
for doc, dist in zip(res["documents"][0], res["distances"][0]):
    print(f"  dist={dist:.3f} | {doc}")


filtered to shipping:
  dist=0.678 | shipping takes 5-7 days
  dist=0.780 | express shipping 2 days


## 8. FAISS: IndexFlatIP (in-memory)
FAISS builds an index over numpy arrays. IndexFlatIP is brute-force inner-product (exact), good for demonstrating the API before ANN indexes.

In [3]:
import faiss

mat = np.array([[1.0, 0.0, 0.1], [0.2, 0.9, 0.1], [0.95, 0.1, 0.0], [0.1, 0.9, 0.0]], dtype="float32")
index = faiss.IndexFlatIP(3)
index.add(mat)
print("faiss ntotal:", index.ntotal)

q = np.array([[0.9, 0.1, 0.0]], dtype="float32")
D, I = index.search(q, 2)
print("nearest indices:", I[0], "  inner-product scores:", D[0])
for i in I[0]:
    print("  faiss hit:", ["return db", "shipping fast", "refund 7d", "express"][i])


faiss ntotal: 4
nearest indices: [0 2]   inner-product scores: [0.9        0.86499995]
  faiss hit: return db
  faiss hit: refund 7d


## 9. Chroma vs FAISS - When to Use What
Both answered similar 'nearest neighbour' questions. Chroma bundles metadata + filtering + persistence; FAISS is a lean, fast numpy index. Neither needs a server, and neither writes to your repo in this ephemeral/in-memory config.

In [4]:
import pandas as pd
cmp = pd.DataFrame([
    ["Chroma", "Local prototype, metadata filters, simple API", "Keep everything in one object"],
    ["FAISS", "High-performance raw vector search", "Brute-force / exact nearest neighbour"],
    ["Pinecone", "Managed cloud, production scale", "No local infra to run (needs account)"],
    ["Qdrant", "High perf + filtering in production", "More moving parts locally"],
], columns=["Tool", "Use when", "Notes"])
print(cmp.to_string(index=False))


    Tool                                      Use when                                 Notes
  Chroma Local prototype, metadata filters, simple API         Keep everything in one object
   FAISS            High-performance raw vector search Brute-force / exact nearest neighbour
Pinecone               Managed cloud, production scale No local infra to run (needs account)
  Qdrant           High perf + filtering in production             More moving parts locally


## 10. ANN: HNSW / IVF
Beyond brute force, ANN indexes trade exactness for speed:
- **HNSW**: build a multi-layer graph; navigate from coarse to fine. Parameters: `M` (connections) and `ef_search` (candidates at query time - higher = better recall, slower).
- **IVF**: partition vectors into `n_lists`; probe `n_probe` partitions.

In [5]:
# Demonstrate the recall vs speed knob conceptually with FAISS's IVF index.
rng = np.random.default_rng(1)
data = rng.normal(size=(500, 8)).astype("float32")
data /= np.linalg.norm(data, axis=1, keepdims=True)

nlist = 10
ivf = faiss.IndexIVFFlat(faiss.IndexFlatL2(8), 8, nlist, faiss.METRIC_INNER_PRODUCT)
ivf.train(data)
ivf.add(data)
print("ntotal:", ivf.ntotal)

def top1_matches(nprobe):
    ivf.nprobe = nprobe
    matched = 0
    for qid in range(0, 500, 25):
        v = data[qid:qid+1]
        D, I = ivf.search(v, 1)
        matched += int(I[0][0] == qid) if ivf.ntotal else 0
    return matched

print("nprobe=1 top1 self-matches:", top1_matches(1), "/ 20")
print("nprobe=10 top1 self-matches:", top1_matches(10), "/ 20")


ntotal: 500
nprobe=1 top1 self-matches: 20 / 20
nprobe=10 top1 self-matches: 20 / 20



## Common Mistakes

- Not tuning index parameters (default HNSW may be slow).
- Ignoring the speed/quality trade-off (ef_search, n_probe).
- Mixing vector dimensions in one collection.
- Not using metadata filtering to narrow the search.

## Debugging

| Symptom | Likely Cause | Fix |
|---|---|---|
| Slow search | Index too small / not built | Build index, tune ef_search |
| Low recall | Index params too aggressive | Increase n_probe / ef_search |
| OOM | dim x count too large | Quantize or disk index |
| Empty metadata filter | Wrong filter syntax | Print metadata; verify values |

## Best Practices

- Start with Chroma for prototyping, migrate later.
- Tune index parameters for speed/quality.
- Use metadata filtering to reduce search space.
- Benchmark recall@k against brute force.

## Hands-On Practice

1. **Basic:** Create a Chroma collection, add 10 vectors, query top-3.
2. **Guided:** Add metadata and compare filtered vs unfiltered queries.
3. **Independent:** Build an index of 1000+ vectors and measure latency.
4. **Realistic:** Compare Chroma and FAISS on the same dataset.
5. **Challenge:** Tune HNSW params and measure the speed-recall trade-off.

## Exit Criteria

- You can explain and build the concept from scratch.
- You can debug the associated failure modes.
- You know when to reach for this tool vs. a plain function.
